This notebook creates precipitation time series from the MSWEP dataset. It uses the basin shapefile to extract precipitation over each basin and then computes the spatial mean to obtain daily precipitation time series.

In [1]:
import os
import xarray as xr
import geopandas as gpd
import rioxarray
import glob
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [2]:
# ------------ Paths ------------
shapefile_path = Path("./caravan_shapefiles/camels/camels_basin_shapes.shp")
basins_path = Path("./basins_excluded.txt")
timeseries_output_dir = Path("./mswep_precip_timeseries")
timeseries_output_dir.mkdir(exist_ok=True)


In [3]:
gdf = gpd.read_file(shapefile_path)
gdf.head()

,gauge_id,geometry
0,camels_01013500,"MULTIPOLYGON (((-68.06259 47.17901, -68.06028 ..."
1,camels_01022500,"POLYGON ((-67.97836 44.6131, -67.98141 44.6143..."
2,camels_01030500,"MULTIPOLYGON (((-68.09162 46.11477, -68.08453 ..."
3,camels_01031500,"MULTIPOLYGON (((-69.31629 45.15325, -69.32144 ..."
4,camels_01047000,"POLYGON ((-70.10847 45.21669, -70.10609 45.213..."


In [4]:
with open(basins_path, "r") as f:
    basins = f.read().splitlines()
basins

['camels_01181000',
 'camels_01411300',
 'camels_01440400',
 'camels_01487000',
 'camels_01491000',
 'camels_01591400',
 'camels_01594950',
 'camels_01605500',
 'camels_01606500',
 'camels_01644000',
 'camels_01664000',
 'camels_01667500',
 'camels_02011400',
 'camels_02014000',
 'camels_02016000',
 'camels_02017500',
 'camels_02018000',
 'camels_02028500',
 'camels_02051500',
 'camels_02053200',
 'camels_02053800',
 'camels_02056900',
 'camels_02059500',
 'camels_02069700',
 'camels_02070000',
 'camels_02074500',
 'camels_02077200',
 'camels_02081500',
 'camels_02092500',
 'camels_02096846',
 'camels_02102908',
 'camels_02111180',
 'camels_02111500',
 'camels_02112120',
 'camels_02112360',
 'camels_02118500',
 'camels_02125000',
 'camels_02128000',
 'camels_02137727',
 'camels_02140991',
 'camels_02143000',
 'camels_02143040',
 'camels_02149000',
 'camels_02152100',
 'camels_02177000',
 'camels_02178400',
 'camels_02193340',
 'camels_02196000',
 'camels_02202600',
 'camels_02212600',


In [5]:
gdf = gdf[gdf["gauge_id"].isin(basins)]
gdf

,gauge_id,geometry
23,camels_01181000,"POLYGON ((-73.04427 42.41436, -73.0383 42.4107..."
31,camels_01411300,"POLYGON ((-74.82836 39.38169, -74.83051 39.379..."
40,camels_01440400,"POLYGON ((-75.19189 41.18651, -75.19333 41.185..."
46,camels_01487000,"MULTIPOLYGON (((-75.51965 38.87137, -75.51541 ..."
47,camels_01491000,"MULTIPOLYGON (((-75.62847 39.00272, -75.62813 ..."
...,...,...
510,camels_09484600,"MULTIPOLYGON (((-110.81922 31.90117, -110.8185..."
547,camels_11124500,"POLYGON ((-119.81038 34.71319, -119.79876 34.7..."
551,camels_11151300,"POLYGON ((-121.01505 36.37416, -121.01598 36.3..."
590,camels_12048000,"POLYGON ((-123.11652 48.03104, -123.11015 48.0..."


In [ ]:
#Hides line: getfattr: /inputs/MSWEP_V280/Past/Daily/1989001.nc: Operation not supported

import os, sys 
sys.stderr = open(os.devnull, "w");

getfattr: /inputs/MSWEP_V280/Past/Daily/1979002.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1979003.nc: Operation not supported


In [7]:
path_data = '/inputs/MSWEP_V280/Past/Daily'

In [13]:
# List all files
all_files = sorted(glob.glob(f'{path_data}/*.nc'))
# all_files[0]

In [12]:
# xr.open_dataset(all_files[1])

In [9]:
# gdf = gpd.read_file(path_shapefiles)
# gdf

In [10]:
# # Reproject if needed
# if gdf.crs != "EPSG:4326":
#     gdf = gdf.to_crs("EPSG:4326")

In [11]:
# Filter by year 1989–2008
files = [f for f in all_files if 1989 <= int(os.path.basename(f)[:4]) <= 2008]
# files = [f for f in all_files if 2009 <= int(os.path.basename(f)[:4]) <= 2019]

print(f"Total files to process: {len(files)}")

Total files to process: 7305


In [12]:
# shapefiles = os.listdir(path_shapefiles)
# shapefiles=shapefiles[1:]

shapefiles=['durazno.zip']

shapefiles

['durazno.zip']

In [13]:
import sys
from collections import defaultdict
from shapely.geometry import box

# Reproject gdf if needed
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")

# Precompute bounding box of ALL basins (with small buffer)
minx, miny, maxx, maxy = gdf.total_bounds
buffer = 0.2  # degrees
bbox = box(minx - buffer, miny - buffer, maxx + buffer, maxy + buffer)

# Prepare a dict to accumulate results per basin
basin_results = defaultdict(list)

# Open each file once and extract precipitation for ALL basins
for f in tqdm(files, desc="Processing files", file=sys.stdout):
    ds = xr.open_dataset(f)
    da = ds["precipitation"]

    # Rename coordinates and assign CRS
    da = da.rename({"lat": "y", "lon": "x"})
    da = da.rio.write_crs("EPSG:4326")

    # Crop to combined bounding box first (fast)
    da_bbox = da.rio.clip([bbox], crs="EPSG:4326", drop=True)

    date = pd.to_datetime(ds.time.values[0])

    for _, row in gdf.iterrows():
        basin_id = row["gauge_id"]
        geometry = row.geometry

        # Clip from the already-cropped small array (much faster)
        da_clip = da_bbox.rio.clip(
            [geometry],
            crs="EPSG:4326",
            drop=True,
            all_touched=True
        )

        mean_val = da_clip.mean(dim=("y", "x"), skipna=True).values.item()
        basin_results[basin_id].append({"date": date, "precip_mm": mean_val})

    ds.close()

# Save results
for basin_id, results in basin_results.items():
    df_results = pd.DataFrame(results)
    df_results = df_results.sort_values("date").set_index("date")
    out_path = timeseries_output_dir / f'{basin_id}_precip.csv'
    df_results.to_csv(out_path)
    print(f"Saved: {out_path}")


Processing files: 100%|██████████| 7305/7305 [5:54:25<00:00,  2.91s/it]  
Saved: mswep_precip_timeseries/camels_01181000_precip.csv
Saved: mswep_precip_timeseries/camels_01411300_precip.csv
Saved: mswep_precip_timeseries/camels_01440400_precip.csv
Saved: mswep_precip_timeseries/camels_01487000_precip.csv
Saved: mswep_precip_timeseries/camels_01491000_precip.csv
Saved: mswep_precip_timeseries/camels_01591400_precip.csv
Saved: mswep_precip_timeseries/camels_01594950_precip.csv
Saved: mswep_precip_timeseries/camels_01605500_precip.csv
Saved: mswep_precip_timeseries/camels_01606500_precip.csv
Saved: mswep_precip_timeseries/camels_01644000_precip.csv
Saved: mswep_precip_timeseries/camels_01664000_precip.csv
Saved: mswep_precip_timeseries/camels_01667500_precip.csv
Saved: mswep_precip_timeseries/camels_02011400_precip.csv
Saved: mswep_precip_timeseries/camels_02014000_precip.csv
Saved: mswep_precip_timeseries/camels_02016000_precip.csv
Saved: mswep_precip_timeseries/camels_02017500_precip.cs

### Joining the different MSWEP files

In [14]:
# --------------- Paths --------------
precip_dir=Path("./precip_timeseries")
output_dir=Path("./mswep_precip_timeseries")

output_dir.mkdir(exist_ok=True)

In [15]:
for i in range (1, 17):
    basin = f"CAMELS_UY_{i}"
    df_1 = pd.read_csv(precip_dir / f"{basin}_precip.csv")
    df_2= pd.read_csv(precip_dir / f"{basin}_precip_2.csv")
    df_combined = pd.concat([df_1, df_2], ignore_index=True)
    df_combined.to_csv(output_dir / f"{basin}_precip.csv", index=False)


FileNotFoundError: [Errno 2] No such file or directory: 'precip_timeseries/CAMELS_UY_1_precip.csv'